In [1]:
%pip install -q transformers pandas torch tqdm

import math
import re
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from tqdm.auto import tqdm

CONFIG = {
    "tokenizer_name": "facebook/bart-base",  # Clean BOS=0, PAD=1, EOS=2 tokens
    "train_path": "samsum-train.csv",
    "val_path": "samsum-validation.csv",
    "max_src_len": 256,
    "max_tgt_len": 96,
    # Model architecture hyperparameters
    "d_model": 384,
    "n_heads": 6,
    "num_enc_layers": 4,
    "num_dec_layers": 4,
    "dropout": 0.1,
    # Training hyperparameters
    "batch_size": 16,
    "epochs": 10,
    "lr": 3e-4,
    "weight_decay": 0.01,
    "label_smoothing": 0.1,
    "save_path": "custom_summarizer.pt",
}

device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
tokenizer = AutoTokenizer.from_pretrained(CONFIG["tokenizer_name"])

print(f"Device: {device} | Vocab size: {tokenizer.vocab_size}")

Note: you may need to restart the kernel to use updated packages.


C:\Users\ranveer\.conda\envs\ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ranveer\.conda\envs\ai\lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ranveer\.cache\huggingface\hub\models--facebook--bart-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mod

Device: cuda | Vocab size: 50265


In [2]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


class SwiGLU(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        hidden_dim = int(d_model * 8 / 3)
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w3(F.silu(self.w1(x)) * self.w2(x)))


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1024):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout_p = dropout

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, query, key, value, attn_mask=None):
        B, L_q, D = query.shape
        L_k = key.size(1)

        q = self.q_proj(query).view(B, L_q, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(key).view(B, L_k, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(value).view(B, L_k, self.n_heads, self.head_dim).transpose(1, 2)

        # PyTorch native Flash Attention
        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout_p if self.training else 0.0,
        )
        out = out.transpose(1, 2).contiguous().view(B, L_q, D)
        return self.out_proj(out)

In [3]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        x = x + self.dropout(self.attn(self.norm1(x), self.norm1(x), self.norm1(x), src_mask))
        x = x + self.ffn(self.norm2(x))
        return x


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm3 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, tgt_mask, src_mask):
        x = x + self.dropout(self.self_attn(self.norm1(x), self.norm1(x), self.norm1(x), tgt_mask))
        x = x + self.dropout(self.cross_attn(self.norm2(x), enc_out, enc_out, src_mask))
        x = x + self.ffn(self.norm3(x))
        return x


class AdvancedSummarizer(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, num_enc, num_dec, dropout, pad_id):
        super().__init__()
        self.pad_id = pad_id
        self.scale = math.sqrt(d_model)

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_enc = SinusoidalPositionalEncoding(d_model)
        self.drop = nn.Dropout(dropout)

        self.encoder = nn.ModuleList([EncoderBlock(d_model, n_heads, dropout) for _ in range(num_enc)])
        self.enc_norm = RMSNorm(d_model)

        self.decoder = nn.ModuleList([DecoderBlock(d_model, n_heads, dropout) for _ in range(num_dec)])
        self.dec_norm = RMSNorm(d_model)

        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embedding.weight  # Weight tying

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def make_masks(self, src, tgt):
        src_mask = (src != self.pad_id).unsqueeze(1).unsqueeze(2)  # (B, 1, 1, S)
        tgt_pad = (tgt != self.pad_id).unsqueeze(1).unsqueeze(2)   # (B, 1, 1, T)
        T = tgt.size(1)
        causal = torch.tril(torch.ones((1, 1, T, T), dtype=torch.bool, device=tgt.device))
        tgt_mask = tgt_pad & causal                                # (B, 1, T, T)
        return src_mask, tgt_mask

    def encode(self, src, src_mask):
        x = self.drop(self.pos_enc(self.embedding(src) * self.scale))
        for layer in self.encoder:
            x = layer(x, src_mask)
        return self.enc_norm(x)

    def decode(self, tgt, enc_out, tgt_mask, src_mask):
        x = self.drop(self.pos_enc(self.embedding(tgt) * self.scale))
        for layer in self.decoder:
            x = layer(x, enc_out, tgt_mask, src_mask)
        return self.lm_head(self.dec_norm(x))

    def forward(self, src, tgt):
        src_mask, tgt_mask = self.make_masks(src, tgt)
        enc_out = self.encode(src, src_mask)
        return self.decode(tgt, enc_out, tgt_mask, src_mask)

    @torch.no_grad()
    def generate(self, src, bos_id, eos_id, max_len=96, beam_size=4, length_penalty=1.2):
        self.eval()
        src_mask = (src != self.pad_id).unsqueeze(1).unsqueeze(2)
        enc_out = self.encode(src, src_mask)

        # Beam search for batch size 1
        beams = [(torch.tensor([[bos_id]], device=src.device), 0.0)]
        completed = []

        for _ in range(max_len):
            candidates = []
            for seq, score in beams:
                if seq[0, -1].item() == eos_id:
                    completed.append((seq, score))
                    continue

                T = seq.size(1)
                causal = torch.tril(torch.ones((1, 1, T, T), dtype=torch.bool, device=src.device))
                logits = self.decode(seq, enc_out, causal, src_mask)[:, -1, :]
                log_probs = F.log_softmax(logits, dim=-1)

                top_probs, top_ids = torch.topk(log_probs, beam_size, dim=-1)
                for i in range(beam_size):
                    next_seq = torch.cat([seq, top_ids[:, i : i + 1]], dim=1)
                    candidates.append((next_seq, score + top_probs[0, i].item()))

            if not candidates:
                break

            # Keep top beams normalized by length penalty
            candidates.sort(key=lambda x: x[1] / (x[0].size(1) ** length_penalty), reverse=True)
            beams = candidates[:beam_size]

        completed.extend(beams)
        completed.sort(key=lambda x: x[1] / (x[0].size(1) ** length_penalty), reverse=True)
        return completed[0][0][0]

In [5]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<.*?>|\r\n|\r|\n", " ", text)
    return re.sub(r"\s+", " ", text).strip()


class SummaryDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_src, max_tgt):
        df = pd.read_csv(csv_path)[["dialogue", "summary"]].dropna().reset_index(drop=True)
        self.dialogues = df["dialogue"].apply(clean_text).tolist()
        self.summaries = df["summary"].apply(clean_text).tolist()
        self.tokenizer = tokenizer
        self.max_src = max_src
        self.max_tgt = max_tgt

    def __len__(self):
        return len(self.dialogues)

    def __getitem__(self, idx):
        src = self.tokenizer(
            self.dialogues[idx],
            max_length=self.max_src,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )["input_ids"].squeeze(0)

        tgt = self.tokenizer(
            self.summaries[idx],
            max_length=self.max_tgt,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )["input_ids"].squeeze(0)

        return src, tgt


train_loader = DataLoader(
    SummaryDataset(CONFIG["train_path"], tokenizer, CONFIG["max_src_len"], CONFIG["max_tgt_len"]),
    batch_size=CONFIG["batch_size"],
    shuffle=True,
)
val_loader = DataLoader(
    SummaryDataset(CONFIG["val_path"], tokenizer, CONFIG["max_src_len"], CONFIG["max_tgt_len"]),
    batch_size=CONFIG["batch_size"],
)

model = AdvancedSummarizer(
    vocab_size=tokenizer.vocab_size,
    d_model=CONFIG["d_model"],
    n_heads=CONFIG["n_heads"],
    num_enc=CONFIG["num_enc_layers"],
    num_dec=CONFIG["num_dec_layers"],
    dropout=CONFIG["dropout"],
    pad_id=tokenizer.pad_token_id,
).to(device)

criterion = nn.CrossEntropyLoss(
    ignore_index=tokenizer.pad_token_id,
    label_smoothing=CONFIG["label_smoothing"],
)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"] * len(train_loader))
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

best_val_loss = float("inf")

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    train_loss = 0.0

    for src, tgt in tqdm(train_loader, desc=f"Epoch {epoch}/{CONFIG['epochs']}"):
        src, tgt = src.to(device), tgt.to(device)
        tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]

        optimizer.zero_grad()
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(src, tgt_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss += loss.item()

    # Validation pass
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(device), tgt.to(device)
            logits = model(src, tgt[:, :-1])
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
            val_loss += loss.item()

    avg_train = train_loss / len(train_loader)
    avg_val = val_loss / len(val_loader)
    print(f"Epoch {epoch} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), CONFIG["save_path"])
        print("  -> Saved new best model")

Epoch 1/10: 100%|██████████| 921/921 [02:45<00:00,  5.56it/s]


Epoch 1 | Train Loss: 6.6040 | Val Loss: 5.9758
  -> Saved new best model


Epoch 2/10: 100%|██████████| 921/921 [02:44<00:00,  5.58it/s]


Epoch 2 | Train Loss: 5.7807 | Val Loss: 5.7000
  -> Saved new best model


Epoch 3/10: 100%|██████████| 921/921 [02:43<00:00,  5.62it/s]


Epoch 3 | Train Loss: 5.4753 | Val Loss: 5.5273
  -> Saved new best model


... (10 outputs left out)

Epoch 9/10: 100%|██████████| 921/921 [02:45<00:00,  5.56it/s]


Epoch 9 | Train Loss: 4.2105 | Val Loss: 5.1010


Epoch 10/10: 100%|██████████| 921/921 [02:46<00:00,  5.53it/s]


Epoch 10 | Train Loss: 4.1720 | Val Loss: 5.0998


In [6]:
model.load_state_dict(torch.load(CONFIG["save_path"], map_location=device))
model.eval()

def summarize(text, max_len=80, beam_size=4):
    cleaned = clean_text(text)
    src = tokenizer(
        cleaned,
        max_length=CONFIG["max_src_len"],
        truncation=True,
        return_tensors="pt",
    )["input_ids"].to(device)

    out_ids = model.generate(
        src,
        bos_id=tokenizer.bos_token_id,
        eos_id=tokenizer.eos_token_id,
        max_len=max_len,
        beam_size=beam_size,
    )
    return tokenizer.decode(out_ids, skip_special_tokens=True)

sample_text = """
Amit: Hey, did you check the server logs for the payment gateway?
Priya: Yes, there's a timeout spike around 2 PM right after the new patch went live.
Amit: Let's roll back to v2.4 immediately and monitor the error rate for 15 minutes.
Priya: Rolling it back now. I'll ping you once the dashboard turns green.
"""

print("Generated Summary:\n", summarize(sample_text))

Generated Summary:
 Alicia is looking for an essay for a new one of a new one. She's going to be there in 10 minutes.
